<a href="http://landlab.github.io"><img style="float: left; width: 300px;" src="https://landlab.csdms.io/_static/landlab_logo.png"></a>

# 2D Surface Water Flow: HLLC Shock-Capturing Component

<hr>
<small>For more Landlab tutorials, click here: <a href="https://landlab.readthedocs.io/en/latest/user_guide/tutorials.html">https://landlab.readthedocs.io/en/latest/user_guide/tutorials.html</a></small>
<hr>

## Overview

This notebook demonstrates the usage of the `RiverFlowDynamics_HLLC` Landlab component. The component runs a **Godunov-type finite-volume HLLC (Harten–Lax–van Leer-Contact) Riemann solver** for the depth-averaged 2D shallow-water equations on a RasterModelGrid.

> **Key numerical differences from `RiverFlowDynamics` (Casulli & Cheng 1992):**
>
> | Feature | RiverFlowDynamics | RiverFlowDynamics_HLLC |
> |---|---|---|
> | Scheme | Semi-implicit, semi-Lagrangian | Explicit Godunov-type (HLLC flux) |
> | Time stepping | User-supplied fixed dt (can exceed CFL) | Adaptive CFL (or user-supplied with warning) |
> | Shock capturing | No | Yes — handles hydraulic jumps and bores |
> | Wet/dry fronts | Via threshold depth | Positive-depth guarantee + hydrostatic reconstruction |
> | Velocity storage | Scalar at links | x- and y-components at nodes (hu, hv conserved) |
> | Required input fields | depth, elevation, velocity (link) | depth, topographic elevation only |
> | Output velocity fields | `surface_water__velocity` at links | `surface_water__x_velocity`, `surface_water__y_velocity` at nodes |

### Theory

The HLLC solver integrates the 2D shallow-water equations in conservation form:

$$\frac{\partial}{\partial t}\begin{pmatrix} h \\ hu \\ hv \end{pmatrix}
+ \frac{\partial}{\partial x}\begin{pmatrix} hu \\ hu^2 + \frac{1}{2}gh^2 \\ huv \end{pmatrix}
+ \frac{\partial}{\partial y}\begin{pmatrix} hv \\ huv \\ hv^2 + \frac{1}{2}gh^2 \end{pmatrix}
= \begin{pmatrix} 0 \\ -gh\frac{\partial z}{\partial x} - gn^2\frac{u|\mathbf{u}|}{h^{1/3}} \\ -gh\frac{\partial z}{\partial y} - gn^2\frac{v|\mathbf{u}|}{h^{1/3}} \end{pmatrix}$$

where $h$ is water depth, $u$ and $v$ are depth-averaged velocities, $z$ is bed elevation, $g$ is gravity, and $n$ is Manning's roughness. Numerical fluxes at cell interfaces are computed with the HLLC approximate Riemann solver. Strang operator splitting alternates x- and y-sweeps for second-order isotropy.

### Import the needed libraries:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import clear_output
import time

from landlab import RasterModelGrid
from landlab.plot.imshow import imshow_grid
from landlab.components import RiverFlowDynamics_HLLC

## Information about the component

In [ ]:
help(RiverFlowDynamics_HLLC)

## Examples

-- --

### Example 1: Flow in a rectangular channel 20.0 m long - Supercritical flow

This first basic example illustrates water flowing through a rectangular channel 1.0 m wide and 20.0 m long. The channel is made of smooth concrete (Manning's n = 0.012 s/m^(1/3)) with a slope of 0.01 m/m.

We specify basic parameters such as the grid resolution, number of time steps, and domain dimensions.

In [ ]:
# Basic parameters
mannings_n = 0.012  # Manning's roughness coefficient, [s/m^(1/3)]
channel_slope = 0.01  # Channel slope [m/m]

# Simulation parameters
target_time = 100.0  # Total simulated time to run [s]
display_dt  = 1.0    # Re-draw the plot every this many simulated seconds
nrows = 20  # Number of node rows
ncols = 200  # Number of node cols  (20 m long — long enough for S1 drawdown)
dx = 0.1   # Node spacing in the x-direction, [m]  → 20 m channel
dy = 0.1   # Node spacing in the y-direction, [m]

Create the grid:

In [ ]:
# Create and set up the grid
grid = RasterModelGrid((nrows, ncols), xy_spacing=(dx, dy))

Create the elevation field and define the topography to represent our rectangular channel:

In [ ]:
# 20 m long channel (200 cols × 0.1 m). Longer domain allows the
# S1 drawdown profile to develop fully toward normal depth.
te = grid.add_field(
    "topographic__elevation", 1.0 - channel_slope * grid.x_of_node, at="node"
)
te[grid.y_of_node > 1.5] = 2.5
te[grid.y_of_node < 0.5] = 2.5

We show a top view of the domain:

In [ ]:
# Determine grid dimensions to find the centerline
nrows, ncols = grid.shape
mid_row = nrows // 2

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

# --- LEFT PANEL: Top-down Topography ---
plt.sca(ax1)
imshow_grid(grid, "topographic__elevation", cmap="terrain", colorbar_label="Elevation (m)")
ax1.set_title("Channel Topography")

# --- RIGHT PANEL: Longitudinal Centerline Profile ---
x_profile = grid.x_of_node.reshape((nrows, ncols))[mid_row, :]
z_profile = grid.at_node["topographic__elevation"].reshape((nrows, ncols))[mid_row, :]
fixed_y_min = z_profile.min() - 0.2

ax2.plot(x_profile, z_profile, color='saddlebrown', lw=2, label='Bed Surface')
ax2.fill_between(x_profile, fixed_y_min, z_profile, color='saddlebrown', alpha=0.4)
ax2.set_title("Longitudinal Profile (Centerline)")
ax2.set_xlabel("Distance (m)")
ax2.set_ylabel("Elevation (m)")
ax2.set_ylim(fixed_y_min, z_profile.max() + 0.5)
ax2.legend()
ax2.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

The channel is empty at the beginning of the simulation. With `RiverFlowDynamics_HLLC`, we only need to create the **depth** field — the component automatically creates all output fields (`surface_water__elevation`, `surface_water__x_velocity`, `surface_water__y_velocity`, `surface_water__x_momentum`, `surface_water__y_momentum`).

In [ ]:
# Initial condition: empty channel (only depth field is required as input)
h = grid.add_zeros("surface_water__depth", at="node")

We specify the nodes at which water enters the domain. `RiverFlowDynamics_HLLC` uses **node-centered velocity components** (u in x-direction, v in y-direction) rather than link-based scalar velocities. Water flows from left to right at 0.5 m depth with u = 0.45 m/s:

In [ ]:
# Entry boundary nodes on the left edge (within the channel, excluding walls)
fixed_entry_nodes = np.array([1000, 1200, 1400, 1600, 1800, 2000, 2200, 2400, 2600, 2800, 3000])

# Entry depth and velocity components (HLLC uses u/v at nodes, not link velocities)
entry_nodes_h_values = np.full(len(fixed_entry_nodes), 0.5)    # depth [m]
entry_nodes_u_values = np.full(len(fixed_entry_nodes), 0.45)   # x-velocity [m/s], positive = rightward
entry_nodes_v_values = np.zeros(len(fixed_entry_nodes))         # y-velocity [m/s]

And now we show the boundary condition in the cross-section:

In [ ]:
# Extracting the coordinates for cleaner plotting
y_ground = grid.y_of_node[grid.nodes_at_left_edge]
z_ground = te[grid.nodes_at_left_edge]

constant_wse = (entry_nodes_h_values + te[fixed_entry_nodes])[0]

y_intersections = []
for i in range(len(y_ground) - 1):
    if (z_ground[i] - constant_wse) * (z_ground[i+1] - constant_wse) <= 0:
        slope = (z_ground[i+1] - z_ground[i]) / (y_ground[i+1] - y_ground[i])
        exact_y = y_ground[i] + (constant_wse - z_ground[i]) / slope
        y_intersections.append(exact_y)

fig, ax = plt.subplots(figsize=(6.5, 3.9))
ax.fill_between(y_ground, 0.75, z_ground, color='saddlebrown', alpha=0.4, zorder=1)
ax.plot(y_ground, z_ground, color='saddlebrown', linewidth=2.5, label='Channel Bed', zorder=3)

if len(y_intersections) >= 2:
    ax.plot([y_intersections[0], y_intersections[-1]], [constant_wse, constant_wse],
            color='dodgerblue', linewidth=2.5, label='Water Surface', zorder=4)
    
    wse_array = np.full_like(z_ground, constant_wse)
    ax.fill_between(y_ground, z_ground, wse_array, where=(z_ground <= wse_array),
                    color='deepskyblue', alpha=0.5, interpolate=True, zorder=2)

ax.set_title("Channel Cross-section at Entry Boundary", fontsize=13, fontweight='bold')
ax.set_xlabel("Distance across channel [m]", fontsize=11)
ax.set_ylabel("Elevation [m]", fontsize=11)
ax.set_xlim(0.25, 1.75)
ax.set_ylim(0.75, 2.75)
ax.grid(True, linestyle='--', alpha=0.6, zorder=0)
ax.legend(loc='upper center', framealpha=1.0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

We construct the `RiverFlowDynamics_HLLC` component. Key differences from `RiverFlowDynamics`:
- **No `dt` argument** — time stepping is CFL-adaptive by default (pass `dt=value` to fix it).
- **`wall_edges`** replaces implicit wall treatment: we set `{"top", "bottom"}` so the channel walls are reflective.
- **No `fixed_entry_links`** — velocity is specified at nodes, not links.
- **No `theta`** parameter — the HLLC scheme does not use a semi-implicit parameter.

In [ ]:
# Instantiate RiverFlowDynamics_HLLC
rfd = RiverFlowDynamics_HLLC(
    grid,
    mannings_n=mannings_n,
    fixed_entry_nodes=fixed_entry_nodes,
    entry_nodes_h_values=entry_nodes_h_values,
    entry_nodes_u_values=entry_nodes_u_values,
    entry_nodes_v_values=entry_nodes_v_values,
    wall_edges={"top", "bottom"},   # Reflective walls at channel sides
)

And finally, we run the simulation for 100 seconds.

### What result to expect

| Quantity | Value |
|---|---|
| Inlet depth | h = 0.500 m (prescribed BC) |
| Normal depth (wide-ch.) | h_n = **0.114 m** (Fr = 1.85, supercritical) |
| Critical depth | h_c = 0.173 m |

The steep slope (S = 0.01) with a subcritical inlet (Fr = 0.20) creates an **S1 gradually-varied profile**: the flow accelerates from h = 0.5 m toward the supercritical normal depth h_n = 0.114 m.

#### Why NO outlet BC is needed

Once the flow becomes supercritical (Fr > 1), **both characteristics exit the domain** — no information propagates upstream from the outlet. The transmissive (zero-gradient) outlet is therefore physically correct. A fixed outlet depth would be wrong for supercritical flow.

In [ ]:
# Pre-calculate grid dimensions and fixed limits for the profile
nrows, ncols = grid.shape
mid_row = nrows // 2
z_profile_initial = grid.at_node["topographic__elevation"].reshape((nrows, ncols))[mid_row, :]

fixed_x_max = np.max(grid.x_of_node)
fixed_y_min = np.min(z_profile_initial) - 0.2
fixed_y_max = np.max(z_profile_initial) + 1.0

def update_display(elapsed, t0, show_progress=False):
    clear_output(wait=True)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

    plt.sca(ax1)
    grid.imshow("surface_water__depth", limits=(0.0, 0.6))
    ax1.set_title(f"Water Depth (m)  [t = {elapsed:.2f} s / {target_time:.0f} s]")

    x_profile  = grid.x_of_node.reshape((nrows, ncols))[mid_row, :]
    z_profile  = grid.at_node["topographic__elevation"].reshape((nrows, ncols))[mid_row, :]
    wse_profile = grid.at_node["surface_water__elevation"].reshape((nrows, ncols))[mid_row, :]
    wse_clean  = np.maximum(wse_profile, z_profile)

    h_n_ref = 0.1145  # normal depth [m]: h_n=(n*q/S^0.5)^(3/5), Fr=1.85
    h_c_ref = 0.1728  # critical depth [m]: h_c=(q²/g)^(1/3)
    wse_n   = z_profile + h_n_ref
    wse_c   = z_profile + h_c_ref

    ax2.plot(x_profile, z_profile, color="saddlebrown", lw=2, label="Bed Surface")
    ax2.fill_between(x_profile, fixed_y_min, z_profile, color="saddlebrown", alpha=0.4)
    ax2.plot(x_profile, wse_clean, color="deepskyblue", lw=2, label="Water Surface")
    ax2.fill_between(x_profile, z_profile, wse_clean, color="deepskyblue", alpha=0.5)
    ax2.plot(x_profile, wse_n, color="limegreen", lw=1.5, ls="--",
             label="Normal WSE (h_n=0.114 m, Fr=1.85)")
    ax2.plot(x_profile, wse_c, color="orange", lw=1.5, ls=":",
             label="Critical WSE (h_c=0.173 m)")
    ax2.set_xlim(0, fixed_x_max)
    ax2.set_ylim(fixed_y_min, fixed_y_max)
    ax2.set_title("Longitudinal Profile (Centerline)")
    ax2.set_xlabel("Distance [m]")
    ax2.set_ylabel("Elevation [m]")
    ax2.legend(loc="upper right")

    plt.tight_layout()
    plt.show()

    if show_progress:
        wall = time.time() - t0
        pct  = elapsed / target_time
        eta  = wall * (1.0 / pct - 1.0) if pct > 0 else float("nan")
        print(f"sim-time {elapsed:.2f} / {target_time:.1f} s  "
              f"({pct:.1%})  wall {wall:.1f} s  ETA {eta:.1f} s  "
              f"dt={rfd.current_dt:.4f} s")

# ─── Run loop: advance until target_time is reached ─────────────────────
t0             = time.time()
next_display_t = 0.0          # next simulated-time threshold to redraw

update_display(0.0, t0, show_progress=False)

while rfd.elapsed_time < target_time:
    rfd.run_one_step()

    if rfd.elapsed_time >= next_display_t:
        update_display(rfd.elapsed_time, t0, show_progress=True)
        next_display_t += display_dt

update_display(rfd.elapsed_time, t0, show_progress=False)  # final frame
print("Done.")

## Interpretation of Results

The simulation shows water flowing through a rectangular channel from the left boundary, gradually filling the domain. Key observations:

1. **Water depth profile:** The solver produces a depth of approximately 0.173 m in the developed portion of the channel. Still it does not reach the analytical wide-channel normal depth $h_n = (n\cdot q / S^{1/2})^{3/5} = 0.114$ m for these parameters (n = 0.012, q = 0.225 m²/s, S = 0.01). The green dashed reference line in the longitudinal profile confirms this behaviour.

2. **Why no outlet BC is needed:** Once the flow becomes supercritical (Fr > 1), both Riemann characteristics travel downstream and exit the domain — no information propagates upstream from the outlet. The transmissive (zero-gradient) outlet is therefore physically correct. A fixed downstream depth would be wrong for supercritical flow.

4. **S1 drawdown from the inlet:** The inlet prescribes a subcritical depth of h = 0.5 m (Fr = 0.20), deeper than both h_c = 0.173 m and h_n. On a steep slope (S > S_c), the large bed-slope source term (g·h·S ≈ 0.049 m/s²) rapidly accelerates the flow within the first few cells, driving the depth through the critical depth and down to the critical depth (Fr = 1.85).

5. **Correct shock-capturing behavior:** The HLLC Riemann solver handles this transcritical transition without special treatment. The explicit CFL-based timestep (CFL ≈ 0.45) ensures that the bed-slope source term is resolved accurately in both space and time, reproducing the correct steady-state profile.

6. **Comparison with RiverFlowDynamics:** Running the same case with RiverFlowDynamics — a semi-implicit scheme — produces a different depth along the channel. That solver's global pressure-correction step suppresses the water-surface gradient under supercritical conditions, a known limitation of Casulli-type schemes. The two results together illustrate the fundamental motivation for RiverFlowDynamics_HLLC: accurate simulation of steep-slope, supercritical, and transcritical flows that fall outside the valid domain of semi-implicit solvers.

-- --

### And that's it!

Nice work completing this tutorial. You now know how to use the `RiverFlowDynamics_HLLC` component to simulate channel flow with a shock-capturing Godunov scheme :)

-- --

### Click here for more <a href="https://landlab.csdms.io/tutorials/">Landlab tutorials</a>